# Split-Attack Notebook for PC-LPUF and iPUF

This notebook implements divide-and-conquer split attacks on **PC-LPUF** and **iPUF** architectures.

The notebook is organized into two modeling approaches:

* **Cells 1–2:** Implement a **Logistic Regression (LR)**-based split attack, following the approach of **Wisiol et al. (CHES 2020)**. This configuration is suitable for PUFs with a relatively small number of lower XOR PUFs (up to approximately 4).
* **Last 2 Cells:** Implement a **Neural Network (NN)**-based split attack, which is more effective for more complex PUFs with more than 4 lower XOR PUFs. The default configuration uses (5, 5) PUF  .

The notebook is designed to be easily configurable. You can modify:

* the PUF architecture (e.g., challenge length, number of upper PUFs, number of lower XOR PUFs),
* the size of the training and testing datasets,

Each section ends with a configuration block where you can adjust the experiment settings and rerun the corresponding attack.


In [ ]:
"""
PC-LPUF Split Attack
====================
Strictly follows the iPUF split attack golden model (Wisiol et al. CHES 2020).

Attack Flow:
  Phase 1 : Randomly guess K_UP upper bits per challenge → obfuscate →
            build LR features → train lower model (~75% label accuracy)
  Round 1+: Use lower model → try all 2^K_UP combos per challenge →
            find which combo matches y_pclpuf → use combo bits as upper APUF labels →
            train K_UP independent APUF models →
            retrain lower with predicted upper bits → repeat

Requirements:
    pip install numpy scipy scikit-learn
"""

import numpy as np
from scipy import sparse
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import accuracy_score
from itertools import product as iterproduct


# ---------------------------------------------------------------------------
# Helper utilities for configuration and dataset swapping
# ---------------------------------------------------------------------------

def build_synthetic_pclpuf_dataset(chal_size, n_levels, n_samples, seed=60):
    """Create synthetic challenges, positions, values, and labels for testing."""
    rng = np.random.default_rng(seed)
    challenges = rng.choice([-1, 1], size=(n_samples, chal_size))
    position = rng.integers(0, chal_size, size=n_samples)
    value = rng.integers(0, n_levels, size=n_samples)
    return challenges, position, value


def build_custom_pclpuf_dataset(challenges, position, value):
    """Wrap user-provided arrays in a simple dictionary for the attack flow."""
    return {
        "challenges": np.asarray(challenges),
        "position": np.asarray(position),
        "value": np.asarray(value),
    }


# ──────────────────────────────────────────────────────────────────────────────
#  1.  REAP-NVM PUF
# ──────────────────────────────────────────────────────────────────────────────


def ReapNVM(num_bits, seed, sigma_proc=0.05):
    # Initialize RNG for reproducible PUF parameter sampling.
    rng = np.random.default_rng(seed)

    chal_length = num_bits
    n_levels = 4

    # Nominal resistance levels used by the ReapNVM delay model.
    R_levels_nom = np.array([10e3,75e3,125e3 ,275e3])

    # Convert delay levels to log domain for Gaussian perturbation.
    log_R_levels_nom = np.log10(R_levels_nom)

    # Allocate arrays for the two logic states and all challenge bits.
    tR = np.zeros((2, n_levels, chal_length))

    # Sample per-cell delays with process variation.
    for row in range(2):
        for stage in range(chal_length):
            # Add Gaussian process variation in log domain.
            log_levels = log_R_levels_nom + sigma_proc * rng.standard_normal(n_levels)

            # Convert back to linear domain.
            levels = 10 ** log_levels

            # RC delay mapping for this stage.
            tR[row, :, stage] = levels * 250e-12

    # Deterministic switching delay added to each path.
    tSW = np.full((2, 2, chal_length), 372.0 / 1e12)

    return 4.0 * tR, 4.0 * tSW


def ReapNVM_evaluate(PUF, challenge, position, value, chunk_size=100_000):
    # Convert challenges from {-1,+1} to {0,1} for the delay model.
    chal      = (challenge + 1) / 2.0
    tR4, tSW4 = PUF
    chalpos   = position.astype(int).flatten()
    chalval   = value.astype(int).flatten()
    N         = chal.shape[0]
    responses = np.zeros(N, dtype=np.int8)

    # Process CRPs in chunks to keep memory use manageable.
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        n   = end - start
        cc  = chal[start:end]
        pc  = chalpos[start:end]
        vc  = chalval[start:end]

        # Build the two path-delay arrays for the current chunk.
        tv1 = np.tile(tR4[0, 0, :], (n, 1)); tv2 = np.tile(tR4[1, 0, :], (n, 1))
        tv1[np.arange(n), pc] = tR4[0, vc, pc]
        tv2[np.arange(n), pc] = tR4[1, vc, pc]

        # Accumulate XOR over the challenge bits.
        c  = np.bitwise_xor.accumulate(cc.astype(np.uint8), axis=1)

        # Compare the two path delays and return the winner as a response bit.
        t1 = np.sum(np.where(c == 0, tv1 + tSW4[0, 0, :], tv2 + tSW4[1, 0, :]), axis=1)
        t2 = np.sum(np.where(c == 0, tv2 + tSW4[0, 1, :], tv1 + tSW4[1, 1, :]), axis=1)
        responses[start:end] = (t1 > t2).astype(np.int8)
    return responses


# ──────────────────────────────────────────────────────────────────────────────
#  2.  APUF
# ──────────────────────────────────────────────────────────────────────────────

def apuf_generate(k, chal_size, seed=0):
    # Draw random APUF weights for the upper-layer PUF.
    return np.random.default_rng(seed).normal(0, 1, (k, chal_size + 1))


def apuf_response(w, Phi):
    # APUF response is determined by the sign of the linear parity model.
    return (Phi @ w <= 0).astype(np.int8)


# ──────────────────────────────────────────────────────────────────────────────
#  3.  XOR Obfuscation
# ──────────────────────────────────────────────────────────────────────────────

def dec_to_bin_vec(x, bitlen):
    # Convert an integer to a bit vector in big-endian order.
    return np.array([(x >> i) & 1 for i in range(bitlen)][::-1], dtype=np.uint8)


def bin_vec_to_dec(bits):
    # Convert a bit vector back to an integer.
    out = 0
    for b in bits: out = (out << 1) | int(b)
    return out


def sliding_window_xor(bits, window_bits):
    # Apply XOR with a sliding window over the bit vector.
    window_bits = window_bits.astype(bits.dtype)
    for start in range(0, bits.shape[0], window_bits.shape[0]):
        end = min(start + window_bits.shape[0], bits.shape[0])
        bits[start:end] ^= window_bits[:end - start]
    return bits


def xor_obfuscate_position_value(position, value, upper_resp):
    """
    upper_resp : (K_UP, N) in {0,1}
    position   : (N,)
    value      : (N,)
    """
    N = position.shape[0]
    pos_out = np.zeros(N, dtype=np.uint32)
    val_out = np.zeros(N, dtype=np.uint32)

    # Obfuscate each position/value pair with the guessed upper-bit pattern.
    for n in range(N):
        window_bits = upper_resp[:, n]
        pos_bits    = sliding_window_xor(dec_to_bin_vec(position[n], 7), window_bits)
        val_bits    = sliding_window_xor(dec_to_bin_vec(value[n],    2), window_bits)
        pos_out[n]  = bin_vec_to_dec(pos_bits)
        val_out[n]  = bin_vec_to_dec(val_bits)
    return pos_out, val_out


# ──────────────────────────────────────────────────────────────────────────────
#  4.  PC-LPUF Evaluate
# ──────────────────────────────────────────────────────────────────────────────

def pclpuf_evaluate(upper_w, lower_pufs, challenges, position, value):
    # Evaluate the full PC-LPUF by first generating upper responses and then obfuscating the lower-layer inputs.
    K_UP = upper_w.shape[0]
    N    = challenges.shape[0]
    Phi  = transform(challenges)
    upper_resp = np.array([apuf_response(upper_w[i], Phi) for i in range(K_UP)])
    pos_eff, val_eff = xor_obfuscate_position_value(position, value, upper_resp)
    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        xor_resp = np.bitwise_xor(xor_resp, ReapNVM_evaluate(puf, challenges, pos_eff, val_eff))
    return xor_resp


def lower_layer_evaluate(lower_pufs, challenges, pos_eff, val_eff):
    """Evaluate lower XOR PUF directly — no obfuscation applied here."""
    N        = challenges.shape[0]
    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        xor_resp = np.bitwise_xor(xor_resp, ReapNVM_evaluate(puf, challenges, pos_eff, val_eff))
    return xor_resp


# ──────────────────────────────────────────────────────────────────────────────
#  5.  Parity Transform + Feature Builder
# ──────────────────────────────────────────────────────────────────────────────

def transform(challenges):
    # Build the parity features used by the logistic regression model.
    N = challenges.shape[0]
    return np.hstack([np.cumprod(challenges, axis=1), np.ones((N, 1))])


def prepare_lr_features(Phi, positions, values, chal_size, n_levels):
    """Build LR feature matrix from parity features + obfuscated pos/val."""
    N         = Phi.shape[0]
    pos       = positions.astype(int)
    val       = values.astype(int)
    delta_idx = pos * n_levels + val
    data      = Phi[np.arange(N), pos]
    X_delta   = sparse.csr_matrix((data, (np.arange(N), delta_idx)),
                                   shape=(N, chal_size * n_levels))
    return sparse.hstack([sparse.csr_matrix(Phi), X_delta], format='csr')


# ──────────────────────────────────────────────────────────────────────────────
#  6.  XOR LR Loss and Gradient
# ──────────────────────────────────────────────────────────────────────────────

def xor_lr_loss_and_grad(w_flat, X_list, y, K, feat_size, lam=1e-4):
    # Logistic loss for a XOR of K linear models.
    N  = X_list[0].shape[0]
    w  = w_flat.reshape(K, feat_size)
    margins   = np.array([X_list[k].dot(w[k]) for k in range(K)])
    signs     = np.sign(margins)
    log_abs   = np.log(np.abs(margins) + 1e-12)
    prod_sign = np.prod(signs, axis=0)
    log_sum   = np.sum(log_abs, axis=0)
    product   = prod_sign * np.exp(np.clip(log_sum, -500, 500))
    p         = np.clip(expit(product), 1e-12, 1 - 1e-12)
    loss      = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    loss     += lam * 0.5 * np.dot(w_flat, w_flat)
    err       = (p - y) / N
    grad      = np.zeros_like(w)
    for k in range(K):
        others  = np.prod(margins[np.arange(K) != k], axis=0)
        grad[k] = X_list[k].T.dot(err * others) + lam * w[k]
    return loss, grad.flatten()


def predict_xor_lr(w_flat, X_list, K, feat_size):
    # Predict labels from the XOR of K logistic-regression submodels.
    w         = w_flat.reshape(K, feat_size)
    margins   = np.array([X_list[k].dot(w[k]) for k in range(K)])
    prod_sign = np.prod(np.sign(margins), axis=0)
    log_sum   = np.sum(np.log(np.abs(margins) + 1e-12), axis=0)
    product   = prod_sign * np.exp(np.clip(log_sum, -500, 500))
    return (product <= 0).astype(np.int8)


def train_xor_lr(X_train, y_train, K, lam=1e-4, max_iter=1000, w0=None, seed=42):
    # Train the XOR LR model with L-BFGS-B.
    feat_size = X_train.shape[1]
    X_sp      = sparse.csr_matrix(X_train)
    X_list    = [X_sp for _ in range(K)]
    if w0 is None:
        np.random.seed(seed)
        w0 = np.random.randn(K * feat_size) * 0.01
    result = minimize(
        fun=xor_lr_loss_and_grad, x0=w0,
        args=(X_list, y_train.astype(float), K, feat_size, lam),
        jac=True, method='L-BFGS-B',
        options={'maxiter': max_iter, 'ftol': 1e-10, 'gtol': 1e-6, 'disp': False}
    )
    return result.x, feat_size


def eval_model(w_flat, X_sp, K, feat_size):
    """Return {0,1} predictions."""
    return predict_xor_lr(w_flat, [X_sp for _ in range(K)], K, feat_size)


# ──────────────────────────────────────────────────────────────────────────────
#  7.  Phase 1 — Random upper bit guess
#      Mirrors interpose_random() in iPUF golden model
# ──────────────────────────────────────────────────────────────────────────────

def interpose_random_pclpuf(challenges, position, value,
                             K_UP, chal_size, n_levels, seed=0):
    """
    Randomly guess K_UP upper APUF bits per challenge — uniform {0,1}.
    Mirrors iPUF: rng.choice([-1, 1]) per challenge.
    Expected ~75% training label accuracy (same math as iPUF).
    """
    N   = challenges.shape[0]
    rng = np.random.default_rng(seed)
    Phi = transform(challenges)

    # Randomly assign upper bits and obfuscate the lower-layer feature inputs.
    guessed_upper = rng.integers(0, 2, size=(K_UP, N)).astype(np.int8)
    pos_chosen, val_chosen = xor_obfuscate_position_value(position, value, guessed_upper)

    return prepare_lr_features(Phi, pos_chosen, val_chosen, chal_size, n_levels)


# ──────────────────────────────────────────────────────────────────────────────
#  8.  Build Upper PUF Training Set
#      Mirrors build_upper_training_set() in iPUF golden model
# ──────────────────────────────────────────────────────────────────────────────

def build_upper_training_set(challenges, position, value, y_pclpuf,
                              w_down, K_d, feat_size_down,
                              chal_size, n_levels, K_UP,
                              block_size=10_000):
    """
    For each challenge:
      1. Try all 2^K_UP upper response combos
      2. Evaluate lower model with each combo's obfuscated (pos_eff, val_eff)
      3. Find all combos where lower model output matches y_pclpuf
      4. Randomly pick ONE correct combo
      5. That combo's K_UP bits are the individual APUF labels

    Returns:
      sel_C     : (N_sel, chal_size)  selected challenges
      sel_resps : (N_sel, K_UP)       individual APUF labels per challenge
    """
    combos    = np.array(list(iterproduct([0, 1], repeat=K_UP)), dtype=np.int8)  # (2^K_UP, K_UP)
    N         = challenges.shape[0]
    sel_chals = []
    sel_resps = []

    for idx in range(0, N, block_size):
        bc  = challenges[idx:idx+block_size]
        bp  = position[idx:idx+block_size]
        bv  = value[idx:idx+block_size]
        br  = y_pclpuf[idx:idx+block_size]   # {0,1}
        bn  = len(bc)
        phi = transform(bc)

        # Evaluate lower model for all 2^K_UP combos → shape (2^K_UP, bn)
        all_preds = np.zeros((len(combos), bn), dtype=np.int8)
        for c_idx, combo in enumerate(combos):
            ur           = np.tile(combo.reshape(K_UP, 1), (1, bn))  # (K_UP, bn)
            pos_c, val_c = xor_obfuscate_position_value(bp, bv, ur)
            X_c          = prepare_lr_features(phi, pos_c, val_c, chal_size, n_levels)
            all_preds[c_idx] = eval_model(w_down, sparse.csr_matrix(X_c), K_d, feat_size_down)

        # For each challenge find combos that match y_pclpuf.
        matches = (all_preds == br[np.newaxis, :])  # (2^K_UP, bn)

        n_insensitive = 0  # all 8 combos correct → challenge insensitive to upper bits
        n_model_wrong = 0  # 0 combos correct → lower model wrong for all combos

        for n in range(bn):
            correct_idxs = np.where(matches[:, n])[0]
            if len(correct_idxs) == 0:
                n_model_wrong += 1
                continue  # lower model wrong for all combos → skip
            if len(correct_idxs) == len(combos):
                n_insensitive += 1
                # Challenge insensitive to upper bits — still include it,
                # pick any combo (all equivalent)

            # Randomly pick one correct combo → its K_UP bits = upper APUF labels.
            chosen_combo = combos[correct_idxs[np.random.randint(len(correct_idxs))]]
            sel_chals.append(bc[n])
            sel_resps.append(chosen_combo)   # (K_UP,)

        if (idx + block_size) % 50_000 == 0:
            print(f"    {min(idx+block_size, N)}/{N} processed, "
                  f"{len(sel_chals)} selected so far | "
                  f"insensitive={n_insensitive} model_wrong={n_model_wrong}")

    if len(sel_chals) == 0:
        return None, None

    return np.array(sel_chals), np.array(sel_resps)  # (N_sel, n), (N_sel, K_UP)


# ──────────────────────────────────────────────────────────────────────────────
#  9.  Full Split Attack
#      Mirrors split_attack() in iPUF golden model
# ──────────────────────────────────────────────────────────────────────────────

def split_attack(upper_w, lower_pufs, chal_size, n_levels, K_UP, K_d,
                 n_train, n_test, max_rounds=5, target_acc=0.90,
                 lam=1e-4, max_iter=500):

    print(f"\n{'='*60}")
    print(f"  PC-LPUF Split Attack")
    print(f"  chal_size={chal_size}, K_UP={K_UP}, K_d={K_d}")
    print(f"  CRPs : train={n_train:,}  test={n_test:,}")
    print(f"{'='*60}")

    # ── Generate CRPs ─────────────────────────────────────────────────────────
    print("\n[1] Generating CRPs...")
    np.random.seed(60)
    challenges_tr = np.random.choice([-1, 1], size=(n_train, chal_size))
    pos_tr        = np.random.randint(0, chal_size, n_train)
    val_tr        = np.random.randint(0, n_levels,  n_train)
    challenges_te = np.random.choice([-1, 1], size=(n_test,  chal_size))
    pos_te        = np.random.randint(0, chal_size, n_test)
    val_te        = np.random.randint(0, n_levels,  n_test)

    # Evaluate the true PC-LPUF responses for train and test data.
    y_train = pclpuf_evaluate(upper_w, lower_pufs, challenges_tr, pos_tr, val_tr)
    y_test  = pclpuf_evaluate(upper_w, lower_pufs, challenges_te, pos_te, val_te)
    print(f"    Response balance: {y_train.mean()*100:.1f}% ones")

    # Ground truth: lower layer WITHOUT obfuscation (original pos/val).
    Phi_te       = transform(challenges_te)
    y_lower_true = lower_layer_evaluate(lower_pufs, challenges_te, pos_te, val_te)

    # ── Phase 1: Train lower model with random upper bit guesses ──────────────
    # Retry up to 10 times if stuck at ~50% — mirrors iPUF paper exactly.
    print(f"\n[2] Phase 1 — Training lower model...")
    print(f"    Randomly guessing {K_UP} upper bits per challenge")
    print(f"    Expected ~75% label accuracy (same math as iPUF)")

    w_down = None; feat_size_down = None; acc_lo = 0.0

    for attempt in range(10):
        print(f"    Attempt {attempt+1}/10...")

        # Spread seeds far apart — same strategy as iPUF retry fix.
        base_seed = attempt * 17

        # Train lower model using randomly guessed upper bits.
        Phi_train = interpose_random_pclpuf(
            challenges_tr, pos_tr, val_tr, K_UP, chal_size, n_levels,
            seed=30 + base_seed
        )
        w_down, feat_size_down = train_xor_lr(
            Phi_train, y_train, K_d, lam=lam, max_iter=max_iter,
            seed=42 + base_seed
        )

        # Check how well the lower model matches the full PC-LPUF with fresh random guesses.
        Phi_check  = interpose_random_pclpuf(
            challenges_te, pos_te, val_te, K_UP, chal_size, n_levels,
            seed=50 + base_seed
        )
        X_check    = [sparse.csr_matrix(Phi_check) for _ in range(K_d)]
        pred_check = predict_xor_lr(w_down, X_check, K_d, feat_size_down)
        acc_check  = accuracy_score(y_test, pred_check)
        acc_check  = max(acc_check, 1 - acc_check)
        print(f"    Attempt {attempt+1} accuracy (vs y_pclpuf): {acc_check*100:.2f}%")

        # Accept if NOT stuck at ~50%.
        if not (0.45 <= acc_check <= 0.55):
            print(f"    ✓ Phase 1 succeeded at attempt {attempt+1}")
            break
        else:
            print(f"    ✗ Stuck at ~50%, retrying...")

    # Evaluate lower model vs real lower layer (no obfuscation).
    X_te = [prepare_lr_features(Phi_te, pos_te, val_te,
                                 chal_size, n_levels) for _ in range(K_d)]
    pred_lo = predict_xor_lr(w_down, X_te, K_d, feat_size_down)
    acc_lo  = max(accuracy_score(y_lower_true, pred_lo),
                  1 - accuracy_score(y_lower_true, pred_lo))
    print(f"    Lower model accuracy (vs real lower layer): {acc_lo*100:.2f}%")
    print(f"    (Paper targets ~75% at this stage)")

    # ── Iterative refinement ──────────────────────────────────────────────────
    w_up = None; feat_size_up = None
    acc_lo2      = acc_lo   # will be updated each round
    acc_up_list  = []       # will be updated each round
    acc_full     = 0.0      # will be updated each round

    for rnd in range(max_rounds):
        print(f"\n[Round {rnd+1}]")

        # ── Build upper training set ──────────────────────────────────────────
        # Try all 2^K_UP combos per challenge → pick correct one randomly.
        print("  Building upper PUF training set (trying all combos)...")
        sel_C, sel_r = build_upper_training_set(
            challenges_tr, pos_tr, val_tr, y_train,
            w_down, K_d, feat_size_down,
            chal_size, n_levels, K_UP
        )

        if sel_C is None or len(sel_C) < 50:
            print("  WARNING: Not enough challenges. Stopping.")
            break

        print(f"  Selected {len(sel_C):,} challenges ({len(sel_C)/n_train*100:.1f}%)")

        # ── Train K_UP independent APUF models (no XOR) ───────────────────────
        Phi_up_sel  = transform(sel_C)
        feat_size_up = Phi_up_sel.shape[1]
        new_w_up    = []
        for i in range(K_UP):
            w0_i   = None if w_up is None else w_up[i]
            w_i, _ = train_xor_lr(
                Phi_up_sel, sel_r[:, i],   # label for APUF i
                K=1, lam=lam, max_iter=max_iter,
                w0=w0_i, seed=43 + rnd + i
            )
            new_w_up.append(w_i)
        w_up = new_w_up

        # ── Evaluate each APUF vs its real response ───────────────────────────
        Phi_te_sp     = sparse.csr_matrix(Phi_te)
        upper_resp_te = np.array([apuf_response(upper_w[i], Phi_te) for i in range(K_UP)])
        acc_up_list   = []
        for i in range(K_UP):
            pred_i = eval_model(w_up[i], Phi_te_sp, 1, feat_size_up)
            acc_i  = max(accuracy_score(upper_resp_te[i], pred_i),
                         1 - accuracy_score(upper_resp_te[i], pred_i))
            acc_up_list.append(acc_i)
        print(f"  Upper APUF accuracies: {[f'{a*100:.1f}%' for a in acc_up_list]}")
        print(f"  Upper mean accuracy  : {np.mean(acc_up_list)*100:.2f}%")

        # ── Retrain lower with predicted upper bits ────────────────────────────
        print("  Retraining lower model with predicted upper bits...")
        Phi_tr  = transform(challenges_tr)
        Phi_tr_sp = sparse.csr_matrix(Phi_tr)

        # Predict each APUF independently → (K_UP, N_train).
        pred_upper_resp = np.array([
            eval_model(w_up[i], Phi_tr_sp, 1, feat_size_up)
            for i in range(K_UP)
        ])

        # Obfuscate with predicted upper bits.
        pos_tr_pred, val_tr_pred = xor_obfuscate_position_value(
            pos_tr, val_tr, pred_upper_resp)
        X_train_new = prepare_lr_features(Phi_tr, pos_tr_pred, val_tr_pred,
                                           chal_size, n_levels)
        w_down, feat_size_down = train_xor_lr(
            X_train_new, y_train, K_d, lam=lam, max_iter=max_iter,
            w0=w_down, seed=44 + rnd
        )

        # ── Evaluate lower model vs real lower layer ───────────────────────────
        X_te2    = [prepare_lr_features(Phi_te, pos_te, val_te,
                                         chal_size, n_levels) for _ in range(K_d)]
        pred_lo2 = predict_xor_lr(w_down, X_te2, K_d, feat_size_down)
        acc_lo2  = max(accuracy_score(y_lower_true, pred_lo2),
                       1 - accuracy_score(y_lower_true, pred_lo2))
        print(f"  Lower model accuracy (vs real lower layer): {acc_lo2*100:.2f}%")

        # ── Full PC-LPUF accuracy ──────────────────────────────────────────────
        # Step 1: predict upper bits using upper model → (K_UP, N_test).
        pred_upper_te = np.array([
            eval_model(w_up[i], sparse.csr_matrix(Phi_te), 1, feat_size_up)
            for i in range(K_UP)
        ])
        # Step 2: obfuscate pos_te/val_te with predicted upper bits.
        pos_te_pred, val_te_pred = xor_obfuscate_position_value(
            pos_te, val_te, pred_upper_te)
        # Step 3: run lower model on obfuscated features.
        X_te_full = [prepare_lr_features(Phi_te, pos_te_pred, val_te_pred,
                                          chal_size, n_levels) for _ in range(K_d)]
        pred_full = predict_xor_lr(w_down, X_te_full, K_d, feat_size_down)
        # Step 4: compare against real PC-LPUF response.
        acc_full  = max(accuracy_score(y_test, pred_full),
                        1 - accuracy_score(y_test, pred_full))
        print(f"  Full PC-LPUF accuracy                     : {acc_full*100:.2f}%")

        if acc_full >= target_acc:
            print(f"\n  ✓ Target {target_acc*100:.0f}% reached at round {rnd+1}!")
            break

    print(f"\n{'─'*60}\n  Done.\n{'─'*60}\n")

    # ── Save results to file ───────────────────────────────────────────────────
    import csv, os
    results_file = f'pclpuf_results_K_UP{K_UP}_Kd{K_d}.csv'
    file_exists  = os.path.isfile(results_file)
    with open(results_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['K_UP', 'K_d', 'chal_size', 'n_levels',
                             'n_train', 'n_test', 'max_rounds',
                             'phase1_lower_acc', 'final_lower_acc',
                             'final_upper_mean_acc', 'final_full_acc'])
        writer.writerow([K_UP, K_d, chal_size, n_levels,
                         n_train, n_test, max_rounds,
                         f'{acc_lo*100:.2f}',
                         f'{acc_lo2*100:.2f}',
                         f'{np.mean(acc_up_list)*100:.2f}' if len(acc_up_list) > 0 else 'N/A',
                         f'{acc_full*100:.2f}'])
    print(f"  Results saved to {results_file}")


# ──────────────────────────────────────────────────────────────────────────────
#  10.  Main
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Main configuration for PC-LPUF experiments.
    # Increase these values to move closer to the paper-style setup.
    chal_size = 128
    n_levels  = 4
    K_UP      = 3
    K_d       = 3
    n_train   = 200_000
    n_test    = 40_000

    np.random.seed(60)
    upper_w    = apuf_generate(K_UP, chal_size, seed=0)
    lower_pufs = [ReapNVM(chal_size, seed=42 + k) for k in range(K_d)]

    split_attack(
        upper_w    = upper_w,
        lower_pufs = lower_pufs,
        chal_size  = chal_size,
        n_levels   = n_levels,
        K_UP       = K_UP,
        K_d        = K_d,
        n_train    = n_train,
        n_test     = n_test,
        max_rounds = 5,
        target_acc = 0.90,
        lam        = 1e-4,
        max_iter   = 500,
    )



  PC-LPUF Split Attack
  chal_size=128, K_UP=3, K_d=3
  CRPs : train=200,000  test=40,000

[1] Generating CRPs...
    Response balance: 49.9% ones

[2] Phase 1 — Training lower model...
    Randomly guessing 3 upper bits per challenge
    Expected ~75% label accuracy (same math as iPUF)
    Attempt 1/10...
    Attempt 1 accuracy (vs y_pclpuf): 63.51%
    ✓ Phase 1 succeeded at attempt 1
    Lower model accuracy (vs real lower layer): 65.03%
    (Paper targets ~75% at this stage)

[Round 1]
  Building upper PUF training set (trying all combos)...
    50000/200000 processed, 29868 selected so far | insensitive=1720 model_wrong=4023
    100000/200000 processed, 59785 selected so far | insensitive=1711 model_wrong=4004
    150000/200000 processed, 89594 selected so far | insensitive=1715 model_wrong=3959
    200000/200000 processed, 119544 selected so far | insensitive=1714 model_wrong=3951
  Selected 119,544 challenges (59.8%)
  Upper APUF accuracies: ['59.3%', '55.0%', '54.4%']
  Upper 

In [ ]:
########ipuf split attack code ends here

"""
Divide-and-Conquer Split Attack on Interpose PUF
=================================================
Replicates Wisiol et al., "Splitting the Interpose PUF: A Novel
Modeling Attack Strategy", CHES 2020.

Attack Flow:
  Phase 1 : Train lower PUF model with random ±1 interpose bit (target ~74%)
  Phase 2 : Use lower model to extract upper PUF training set → train upper model
  Iterate : Retrain lower with predicted interpose bits → retrain upper → ...
  Until   : Overall iPUF accuracy >= 95%

Usage:
    python ipuf_split_attack_full.py

Requirements:
    pip install numpy scipy scikit-learn pypuf
"""

import numpy as np
from numpy.random import RandomState
from scipy import sparse
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import accuracy_score

from pypuf.simulation import InterposePUF
from pypuf.io import random_inputs


# ──────────────────────────────────────────────────────────────────────────────
#  1.  ATF (Parity) Transform  — matches pypuf's transform_atf
# ──────────────────────────────────────────────────────────────────────────────

def att(challenges):
    """
    Apply ATF transform IN-PLACE on challenges of shape (N, n).
    Equivalent to pypuf's LTFArray.att().
    Φ_i = prod_{j=i}^{n-1} c_j  (reverse cumulative product).
    """
    N, n = challenges.shape
    # Reverse cumulative product to match the ATF feature construction.
    for i in range(n - 2, -1, -1):
        challenges[:, i] *= challenges[:, i + 1]


def transform_atf(challenges):
    """
    Return ATF-transformed features (copy, not in-place) with bias.
    Input:  (N, n)   in {-1,+1}
    Output: (N, n+1)
    """
    c = challenges.astype(np.float64).copy()
    att(c)
    return np.hstack([c, np.ones((c.shape[0], 1))])


# ──────────────────────────────────────────────────────────────────────────────
#  2.  XOR LR Loss and Gradient
# ──────────────────────────────────────────────────────────────────────────────

def xor_lr_loss_and_grad(w_flat, X_list, y, K, feat_size, lam=1e-4):
    # Logistic loss for the XOR combination of K submodels.
    N = X_list[0].shape[0]
    w = w_flat.reshape(K, feat_size)

    # Compute margin for each submodel and combine them multiplicatively.
    margins = np.array([X_list[k].dot(w[k]) for k in range(K)])  # (K, N)
    product = np.prod(margins, axis=0)                             # (N,)

    p    = np.clip(expit(product), 1e-12, 1 - 1e-12)
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    loss += lam * 0.5 * np.dot(w_flat, w_flat)

    err  = (p - y) / N
    grad = np.zeros_like(w)
    for k in range(K):
        others  = np.prod(margins[np.arange(K) != k], axis=0)
        grad[k] = X_list[k].T.dot(err * others) + lam * w[k]

    return loss, grad.flatten()


def predict_xor_lr(w_flat, X_list, K, feat_size):
    # Predict class labels from the XOR of the submodel outputs.
    w       = w_flat.reshape(K, feat_size)
    margins = np.array([X_list[k].dot(w[k]) for k in range(K)])
    product = np.prod(margins, axis=0)
    return (product <= 0).astype(np.int8)


def train_xor_lr(X_train, y_train, K, lam=1e-4, max_iter=1000, w0=None, seed=42):
    """Train XOR LR model, return optimal weights."""
    feat_size    = X_train.shape[1]
    X_train_sp   = sparse.csr_matrix(X_train)
    X_train_list = [X_train_sp for _ in range(K)]

    if w0 is None:
        np.random.seed(seed)
        w0 = np.random.randn(K * feat_size) * 0.01

    result = minimize(
        fun=xor_lr_loss_and_grad,
        x0=w0,
        args=(X_train_list, y_train.astype(float), K, feat_size, lam),
        jac=True,
        method='L-BFGS-B',
        options={'maxiter': max_iter, 'ftol': 1e-10, 'gtol': 1e-6, 'disp': False}
    )
    return result.x, feat_size


def eval_model(w_flat, X, K, feat_size):
    """Return {-1,+1} predictions (pypuf convention)."""
    X_sp   = [sparse.csr_matrix(X) for _ in range(K)]
    pred01 = predict_xor_lr(w_flat, X_sp, K, feat_size)
    return 1 - 2 * pred01   # {0→+1, 1→-1}


# ──────────────────────────────────────────────────────────────────────────────
#  3.  Challenge Manipulation
# ──────────────────────────────────────────────────────────────────────────────

def interpose(challenges, bits, pos):
    """
    Insert bits column at position pos.
    challenges : (N, n)  in {-1,+1}
    bits       : (N, 1)  in {-1,+1}
    returns    : (N, n+1)
    """
    N = challenges.shape[0]
    bits = np.array(bits).reshape(N, 1)
    return np.concatenate([challenges[:, :pos], bits, challenges[:, pos:]], axis=1)


def interpose_random(challenges, pos, seed=0):
    """Insert random ±1 interpose bit."""
    N    = challenges.shape[0]
    rng  = RandomState(seed)
    bits = rng.choice([-1, 1], size=(N, 1)).astype(np.int8)
    return interpose(challenges, bits, pos)


# ──────────────────────────────────────────────────────────────────────────────
#  4.  Build Upper PUF Training Set  (from paper's _get_model_up)
# ──────────────────────────────────────────────────────────────────────────────

def build_upper_training_set(challenges, responses, w_down, K_down, feat_size_down, pos):
    """
    For each challenge C:
      - Evaluate lower model with interpose bit = +1 → r_p1
      - Evaluate lower model with interpose bit = -1 → r_m1
      - Keep challenge only if r_p1 != r_m1  (challenge depends on interpose bit)
      - Upper label = r_p1 * r_iPUF  (the bit that matched the real response)

    Returns: (selected_challenges, selected_responses) in {-1,+1}
    """
    N         = challenges.shape[0]
    block_size = 100_000
    sel_chals  = []
    sel_resps  = []

    for idx in range(0, N, block_size):
        block_c = challenges[idx:idx+block_size]
        block_r = responses[idx:idx+block_size]
        block_n = len(block_c)

        # Extend the challenges with both possible interpose bits.
        c_p1 = interpose(block_c, np.ones( (block_n, 1), dtype=np.int8), pos)
        c_m1 = interpose(block_c, -np.ones((block_n, 1), dtype=np.int8), pos)

        # Evaluate lower model on both interpose-bit choices.
        Phi_p1 = transform_atf(c_p1.astype(np.float64))
        Phi_m1 = transform_atf(c_m1.astype(np.float64))
        r_p1   = eval_model(w_down, Phi_p1, K_down, feat_size_down)  # {-1,+1}
        r_m1   = eval_model(w_down, Phi_m1, K_down, feat_size_down)

        # Keep only challenges where response depends on interpose bit.
        unequal = (r_p1 != r_m1)
        if unequal.sum() == 0:
            continue

        # Upper label: r_p1 * r_iPUF  (from paper eq.)
        upper_labels = r_p1[unequal] * block_r[unequal]

        sel_chals.append(block_c[unequal])
        sel_resps.append(upper_labels)

    if len(sel_chals) == 0:
        return None, None

    return np.vstack(sel_chals), np.concatenate(sel_resps)


# ──────────────────────────────────────────────────────────────────────────────
#  5.  Accuracy Evaluation
# ──────────────────────────────────────────────────────────────────────────────

def ipuf_accuracy(puf, w_down, w_up, K_down, K_up,
                  feat_size_down, feat_size_up, pos, n_bits, n_eval=10_000, seed=99):
    """Evaluate full iPUF model accuracy."""
    C      = random_inputs(n=n_bits, N=n_eval, seed=seed).astype(np.float64)
    r_true = puf.eval(C.astype(np.int8))   # {-1,+1}

    # Predict upper bit and insert it into the challenge.
    Phi_up  = transform_atf(C)
    b_pred  = eval_model(w_up, Phi_up, K_up, feat_size_up)  # {-1,+1}
    C_star  = interpose(C, b_pred.reshape(-1, 1), pos)
    Phi_lo  = transform_atf(C_star)
    r_pred  = eval_model(w_down, Phi_lo, K_down, feat_size_down)

    acc = accuracy_score(r_true, r_pred)
    return max(acc, 1 - acc)


# ──────────────────────────────────────────────────────────────────────────────
#  6.  Full Split Attack
# ──────────────────────────────────────────────────────────────────────────────

def split_attack(puf, n_bits, k_up, k_lo, n_train, n_test,
                 max_rounds=5, target_acc=0.95, lam=1e-4, max_iter=1000, puf_id=0):

    pos = n_bits // 2

    print(f"\n{'='*60}")
    print(f"  iPUF Split Attack — Wisiol et al. CHES 2020")
    print(f"  n={n_bits}, k_up={k_up}, k_lo={k_lo}, pos={pos}")
    print(f"  CRPs : train={n_train:,}  test={n_test:,}")
    print(f"{'='*60}")

    # ── Generate CRPs ─────────────────────────────────────────────────────────
    print("\n[1] Generating CRPs...")
    C_train = random_inputs(n=n_bits, N=n_train, seed=10 + puf_id).astype(np.int8)
    C_test  = random_inputs(n=n_bits, N=n_test,  seed=20 + puf_id).astype(np.int8)

    # pypuf: 0→+1, 1→-1, so eval returns {-1,+1}.
    r_train = puf.eval(C_train)   # {-1,+1}
    r_test  = puf.eval(C_test)

    # Convert to {0,1} for LR loss.
    y_train = ((1 - r_train) // 2).astype(np.int8)
    y_test  = ((1 - r_test)  // 2).astype(np.int8)
    print(f"    Response balance: {y_train.mean()*100:.1f}% ones")

    # ── Phase 1: Train lower model with random interpose bit ──────────────────
    print("\n[2] Phase 1 — Training lower model (random interpose bit)...")
    C_train_ext = interpose_random(C_train.astype(np.float64), pos, seed=30 + puf_id)
    Phi_train   = transform_atf(C_train_ext)   # (N, n+2)

    w_down, feat_size_down = train_xor_lr(
        Phi_train, y_train, k_lo, lam=lam, max_iter=max_iter, seed=42 + puf_id
    )

    # Evaluate lower model using the REAL interpose bit versus puf.down directly.
    b_real       = puf.up.eval(C_test)                                     # {-1,+1} real upper bit
    C_star_test  = interpose(C_test.astype(np.float64),
                             b_real.reshape(-1, 1), pos)                   # (N, n+1) real C*
    r_down_true  = puf.down.eval(C_star_test.astype(np.int8))             # {-1,+1} real lower response
    y_down_true  = ((1 - r_down_true) // 2).astype(np.int8)               # {0,1}
    Phi_test_lo  = transform_atf(C_star_test)
    X_test_sp    = [sparse.csr_matrix(Phi_test_lo) for _ in range(k_lo)]
    pred_lo      = predict_xor_lr(w_down, X_test_sp, k_lo, feat_size_down)
    acc_lo       = max(accuracy_score(y_down_true, pred_lo),
                       1 - accuracy_score(y_down_true, pred_lo))
    print(f"    Lower model accuracy (vs puf.down, real bit): {acc_lo*100:.2f}%")

    # ── Iterative refinement ──────────────────────────────────────────────────
    w_up       = None
    feat_size_up = None

    for rnd in range(max_rounds):
        print(f"\n[Round {rnd+1}]")

        # ── Train upper model ─────────────────────────────────────────────────
        print("  Building upper PUF training set...")
        sel_C, sel_r = build_upper_training_set(
            C_train.astype(np.float64), r_train,
            w_down, k_lo, feat_size_down, pos
        )

        if sel_C is None or len(sel_C) < 50:
            print("  WARNING: Not enough challenges for upper model. Stopping.")
            break

        print(f"  Selected {len(sel_C):,} challenges for upper model "
              f"({len(sel_C)/n_train*100:.1f}% of training set)")

        # Convert upper labels {-1,+1} → {0,1} for training.
        y_up = ((1 - sel_r) // 2).astype(np.int8)

        Phi_up = transform_atf(sel_C)
        w_up, feat_size_up = train_xor_lr(
            Phi_up, y_up, k_up, lam=lam, max_iter=max_iter,
            w0=None if w_up is None else w_up,
            seed=43 + puf_id + rnd
        )

        # Evaluate upper model.
        Phi_up_test = transform_atf(C_test.astype(np.float64))
        X_up_sp     = [sparse.csr_matrix(Phi_up_test) for _ in range(k_up)]
        pred_up     = predict_xor_lr(w_up, X_up_sp, k_up, feat_size_up)
        r_up_true   = puf.up.eval(C_test.astype(np.int8))
        y_up_true   = ((1 - r_up_true) // 2).astype(np.int8)
        acc_up      = max(accuracy_score(y_up_true, pred_up),
                          1 - accuracy_score(y_up_true, pred_up))
        print(f"  Upper model accuracy : {acc_up*100:.2f}%")

        # ── Retrain lower model with predicted interpose bits ─────────────────
        print("  Retraining lower model with predicted interpose bits...")
        b_pred      = eval_model(w_up,
                                 transform_atf(C_train.astype(np.float64)),
                                 k_up, feat_size_up)                    # {-1,+1}
        C_train_new = interpose(C_train.astype(np.float64),
                                b_pred.reshape(-1, 1), pos)             # (N, n+1)
        Phi_new     = transform_atf(C_train_new)

        w_down, feat_size_down = train_xor_lr(
            Phi_new, y_train, k_lo, lam=lam, max_iter=max_iter,
            w0=w_down, seed=44 + puf_id + rnd
        )

        # Evaluate lower model versus puf.down using the real interpose bit.
        b_real_rnd      = puf.up.eval(C_test)
        C_star_rnd      = interpose(C_test.astype(np.float64),
                                    b_real_rnd.reshape(-1, 1), pos)
        r_down_rnd      = puf.down.eval(C_star_rnd.astype(np.int8))
        y_down_rnd      = ((1 - r_down_rnd) // 2).astype(np.int8)
        Phi_lo_rnd      = transform_atf(C_star_rnd)
        X_lo_rnd        = [sparse.csr_matrix(Phi_lo_rnd) for _ in range(k_lo)]
        pred_lo_rnd     = predict_xor_lr(w_down, X_lo_rnd, k_lo, feat_size_down)
        acc_lo_rnd      = max(accuracy_score(y_down_rnd, pred_lo_rnd),
                              1 - accuracy_score(y_down_rnd, pred_lo_rnd))
        print(f"  Lower model accuracy (vs puf.down): {acc_lo_rnd*100:.2f}%")

        # ── Evaluate full iPUF model ───────────────────────────────────────────
        acc_total = ipuf_accuracy(
            puf, w_down, w_up, k_lo, k_up,
            feat_size_down, feat_size_up, pos, n_bits
        )
        print(f"  Full iPUF accuracy   : {acc_total*100:.2f}%")

        if acc_total >= target_acc:
            print(f"\n  ✓ Target accuracy {target_acc*100:.0f}% reached at round {rnd+1}!")
            break

    # ── Final summary ──────────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    if w_up is not None:
        final_acc = ipuf_accuracy(
            puf, w_down, w_up, k_lo, k_up,
            feat_size_down, feat_size_up, pos, n_bits, n_eval=20_000
        )
        print(f"  Final iPUF accuracy  : {final_acc*100:.2f}%")
    else:
        print("  Attack did not reach upper model training stage.")
    print(f"{'─'*60}\n")


# ──────────────────────────────────────────────────────────────────────────────
#  7.  Main
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    num_bits = 128
    k_up     = 4
    k_lo     = 3
    puf_id   = 0
    N_TRAIN  = 200_000
    N_TEST   =  80_000

    puf = InterposePUF(
        n=num_bits,
        k_up=k_up,
        k_down=k_lo,
        seed=1 + puf_id,
        noisiness=0,
    )

    split_attack(
        puf        = puf,
        n_bits     = num_bits,
        k_up       = k_up,
        k_lo       = k_lo,
        n_train    = N_TRAIN,
        n_test     = N_TEST,
        max_rounds = 5,
        target_acc = 0.95,
        lam        = 1e-4,
        max_iter   = 1000,
        puf_id     = puf_id,
    )


In [ ]:
"""
iPUF Split Attack — MLP version for k=5
=========================================
Uses MLP (PyTorch) instead of LR for Phase 1 lower model training as LR fail to converge.
Follows Wisiol et al. CHES 2020 split attack structure.

Attack Flow:
  Phase 1 : Train lower model with MLP on random ±1 interpose bit
  Round 1+: Build upper training set → train upper with MLP →
            retrain lower with predicted bit → repeat

Requirements:
    pip install numpy torch pypuf scikit-learn
"""

import numpy as np
from numpy.random import RandomState
from scipy import sparse
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from pypuf.simulation import InterposePUF
from pypuf.io import random_inputs


# ──────────────────────────────────────────────────────────────────────────────
#  1.  ATF Transform
# ──────────────────────────────────────────────────────────────────────────────

def att(challenges):
    N, n = challenges.shape
    for i in range(n - 2, -1, -1):
        challenges[:, i] *= challenges[:, i + 1]

def transform_atf(challenges):
    c = challenges.astype(np.float64).copy()
    att(c)
    return np.hstack([c, np.ones((c.shape[0], 1))])

def arbiter_features_binary(challenges):
    # challenges already in {-1, 1}
    phi = np.flip(np.cumprod(np.flip(challenges, axis=1), axis=1), axis=1)
    return np.hstack([phi, np.ones((phi.shape[0], 1))])


# ──────────────────────────────────────────────────────────────────────────────
#  2.  MLP Training & Evaluation  (user's exact code)
# ──────────────────────────────────────────────────────────────────────────────

def train_mlp_puf(X, y, net=[128, 32, 16], lr=0.001, epochs=20, bs=1000,
                  seed=1, early_stop=None):
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("    Using device:", device)

    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    dataset = TensorDataset(X_t, y_t)
    loader  = DataLoader(dataset, batch_size=bs, shuffle=True)

    layers = []
    in_dim = X.shape[1]
    for h in net:
        layers.append(nn.Linear(in_dim, h))
        layers.append(nn.ReLU())
        in_dim = h
    layers.append(nn.Linear(in_dim, 1))
    layers.append(nn.Sigmoid())

    model   = nn.Sequential(*layers).to(device)
    opt     = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()

        acc = evaluate_mlp(model, X_t, y_t, device)
        print(f"    Epoch {epoch+1}/{epochs} | Loss={total_loss:.4f} | Acc={acc:.4f}")

        if early_stop is not None and acc >= early_stop:
            print("    Early stop triggered.")
            break

    return model


def evaluate_mlp(model, X_t, y_t, device, batch_size=2048):
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb    = torch.tensor(X_t[i:i+batch_size], dtype=torch.float32).to(device)
            yb    = torch.tensor(y_t[i:i+batch_size], dtype=torch.float32).view(-1, 1)
            preds = model(xb).cpu()
            preds = (preds > 0.5).int()
            preds = 2 * preds - 1
            yb    = 2 * yb.int() - 1
            correct += (preds == yb).sum().item()
            total   += yb.size(0)
    return correct / total


def predict_mlp(model, X, device, batch_size=2048):
    """Return {0,1} predictions for numpy array X."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            p  = model(xb).cpu().numpy()
            preds.append((p > 0.5).astype(np.int8))
    return np.vstack(preds).flatten()


# ──────────────────────────────────────────────────────────────────────────────
#  3.  Challenge Manipulation
# ──────────────────────────────────────────────────────────────────────────────

def interpose(challenges, bits, pos):
    N    = challenges.shape[0]
    bits = np.array(bits).reshape(N, 1)
    return np.concatenate([challenges[:, :pos], bits, challenges[:, pos:]], axis=1)

def interpose_random(challenges, pos, seed=0):
    N   = challenges.shape[0]
    rng = RandomState(seed)
    bits = rng.choice([-1, 1], size=(N, 1)).astype(np.int8)
    return interpose(challenges, bits, pos)


# ──────────────────────────────────────────────────────────────────────────────
#  4.  Build Upper Training Set
# ──────────────────────────────────────────────────────────────────────────────

def build_upper_training_set_mlp(challenges, responses, model_down, pos, device,
                                   block_size=100_000):
    """
    Same logic as LR version but uses MLP to evaluate lower model.
    responses in {-1,+1}
    """
    N         = challenges.shape[0]
    sel_chals = []
    sel_resps = []

    for idx in range(0, N, block_size):
        block_c = challenges[idx:idx+block_size]
        block_r = responses[idx:idx+block_size]
        block_n = len(block_c)

        c_p1 = interpose(block_c, np.ones( (block_n, 1), dtype=np.int8), pos)
        c_m1 = interpose(block_c, -np.ones((block_n, 1), dtype=np.int8), pos)

        Phi_p1 = transform_atf(c_p1.astype(np.float64))
        Phi_m1 = transform_atf(c_m1.astype(np.float64))

        # MLP predicts {0,1} → convert to {-1,+1}
        r_p1_01 = predict_mlp(model_down, Phi_p1, device)
        r_m1_01 = predict_mlp(model_down, Phi_m1, device)
        r_p1 = 1 - 2 * r_p1_01   # {0→+1, 1→-1}
        r_m1 = 1 - 2 * r_m1_01

        unequal = (r_p1 != r_m1)
        if unequal.sum() == 0:
            continue

        # Upper label: r_p1 * r_iPUF (paper eq.) → in {-1,+1}
        upper_labels = r_p1[unequal] * block_r[unequal]

        sel_chals.append(block_c[unequal])
        sel_resps.append(upper_labels)

    if len(sel_chals) == 0:
        return None, None

    return np.vstack(sel_chals), np.concatenate(sel_resps)


# ──────────────────────────────────────────────────────────────────────────────
#  5.  Full iPUF Accuracy
# ──────────────────────────────────────────────────────────────────────────────

def ipuf_accuracy_mlp(puf, model_down, model_up, pos, n_bits, device,
                       n_eval=10_000, seed=99):
    C      = random_inputs(n=n_bits, N=n_eval, seed=seed).astype(np.float64)
    r_true = puf.eval(C.astype(np.int8))   # {-1,+1}
    y_true = ((1 - r_true) // 2).astype(np.int8)  # {0,1}

    # Predict upper bit
    Phi_up = transform_atf(C)
    b_pred_01 = predict_mlp(model_up, Phi_up, device)   # {0,1}
    b_pred    = 1 - 2 * b_pred_01                        # {-1,+1}

    # Build extended challenge
    C_star = interpose(C, b_pred.reshape(-1, 1), pos)
    Phi_lo = transform_atf(C_star)
    y_pred = predict_mlp(model_down, Phi_lo, device)

    acc = accuracy_score(y_true, y_pred)
    return max(acc, 1 - acc)


# ──────────────────────────────────────────────────────────────────────────────
#  6.  Full Split Attack with MLP
# ──────────────────────────────────────────────────────────────────────────────

def split_attack_mlp(puf, n_bits, k_up, k_lo, n_train, n_test,
                     max_rounds=5, target_acc=0.95, puf_id=0,
                     net=[256, 128, 64], lr=0.001, epochs=30, bs=1000,
                     early_stop=0.99):

    pos    = n_bits // 2
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{'='*60}")
    print(f"  iPUF Split Attack — MLP (k={k_lo})")
    print(f"  n={n_bits}, k_up={k_up}, k_lo={k_lo}")
    print(f"  CRPs : train={n_train:,}  test={n_test:,}")
    print(f"  MLP  : {net}  lr={lr}  epochs={epochs}")
    print(f"{'='*60}")

    # ── Generate CRPs ─────────────────────────────────────────────────────────
    print("\n[1] Generating CRPs...")
    C_train = random_inputs(n=n_bits, N=n_train, seed=10 + puf_id).astype(np.int8)
    C_test  = random_inputs(n=n_bits, N=n_test,  seed=20 + puf_id).astype(np.int8)

    r_train = puf.eval(C_train)   # {-1,+1}
    r_test  = puf.eval(C_test)
    y_train = ((1 - r_train) // 2).astype(np.int8)
    y_test  = ((1 - r_test)  // 2).astype(np.int8)
    print(f"    Response balance: {y_train.mean()*100:.1f}% ones")

    # ── Phase 1: Train lower model with MLP + retry ───────────────────────────
    print("\n[2] Phase 1 — Training lower model with MLP...")
    model_down = None

    for attempt in range(10):
        print(f"\n    Attempt {attempt+1}/10")
        base_seed = 271828 + attempt

        C_train_ext = interpose_random(C_train.astype(np.float64), pos,
                                       seed=30 + base_seed)
        Phi_train   = transform_atf(C_train_ext)

        model_down = train_mlp_puf(
            X          = Phi_train,
            y          = y_train,
            net        = net,
            lr         = lr,
            epochs     = epochs,
            bs         = bs,
            seed       = base_seed,
            early_stop = 0.74,   # Phase 1 only: stop at 74% (mirrors paper)
        )

        # Check 1: combined model (lower + random upper) vs full iPUF — used for retry decision
        # mirrors paper: model_ipuf.down = model_down; test_set_accuracy = approx_dist(model_ipuf, simulation)
        C_test_ext   = interpose_random(C_test.astype(np.float64), pos, seed=40 + base_seed)
        Phi_test_ext = transform_atf(C_test_ext)
        X_t_check    = torch.from_numpy(Phi_test_ext.astype(np.float32))
        y_t_check    = torch.from_numpy(y_test.astype(np.float32))
        acc_check    = evaluate_mlp(model_down, X_t_check, y_t_check, device)
        acc_check    = max(acc_check, 1 - acc_check)
        print(f"    Attempt {attempt+1} accuracy (random bit vs full iPUF): {acc_check*100:.2f}%")

        # Check 2: lower layer alone with REAL upper bit — print only, not used for retry
        b_real      = puf.up.eval(C_test)
        C_star_test = interpose(C_test.astype(np.float64), b_real.reshape(-1, 1), pos)
        r_down_true = puf.down.eval(C_star_test.astype(np.int8))
        y_down_true = ((1 - r_down_true) // 2).astype(np.int8)
        Phi_test_lo = transform_atf(C_star_test)
        X_t_real    = torch.from_numpy(Phi_test_lo.astype(np.float32))
        y_t_real    = torch.from_numpy(y_down_true.astype(np.float32))
        acc_real    = evaluate_mlp(model_down, X_t_real, y_t_real, device)
        acc_real    = max(acc_real, 1 - acc_real)
        print(f"    Attempt {attempt+1} accuracy (vs puf.down, real bit): {acc_real*100:.2f}%  [info only]")

        if not (0.45 <= acc_check <= 0.55):
            print(f"    ✓ Phase 1 succeeded at attempt {attempt+1}")
            break
        else:
            print(f"    ✗ Stuck at ~50%, retrying...")

    # Final lower model accuracy for reporting
    acc_lo = acc_real  # vs puf.down with real bit

    # ── Iterative refinement ──────────────────────────────────────────────────
    model_up = None

    for rnd in range(max_rounds):
        print(f"\n[Round {rnd+1}]")

        # Build upper training set
        print("  Building upper PUF training set...")
        sel_C, sel_r = build_upper_training_set_mlp(
            C_train.astype(np.float64), r_train,
            model_down, pos, device
        )

        if sel_C is None or len(sel_C) < 50:
            print("  WARNING: Not enough challenges. Stopping.")
            break

        print(f"  Selected {len(sel_C):,} ({len(sel_C)/n_train*100:.1f}%)")

        # Train upper model
        Phi_up_sel = transform_atf(sel_C)
        y_up       = ((1 - sel_r) // 2).astype(np.int8)  # {-1,+1}→{0,1}

        model_up = train_mlp_puf(
            X          = Phi_up_sel,
            y          = y_up,
            net        = net,
            lr         = lr,
            epochs     = epochs,
            bs         = bs,
            seed       = 43 + rnd,
            early_stop = None,   # no early stop for upper model
        )

        # Evaluate upper model
        Phi_up_test = transform_atf(C_test.astype(np.float64))
        r_up_true   = puf.up.eval(C_test.astype(np.int8))
        y_up_true   = ((1 - r_up_true) // 2).astype(np.int8)
        X_up_t      = torch.tensor(Phi_up_test, dtype=torch.float32)
        y_up_t      = torch.tensor(y_up_true,   dtype=torch.float32)
        acc_up      = evaluate_mlp(model_up, X_up_t, y_up_t, device)
        acc_up      = max(acc_up, 1 - acc_up)
        print(f"  Upper model accuracy : {acc_up*100:.2f}%")

        # Retrain lower with predicted upper bits
        print("  Retraining lower model...")
        Phi_tr_up = transform_atf(C_train.astype(np.float64))
        b_pred_01 = predict_mlp(model_up, Phi_tr_up, device)
        b_pred    = 1 - 2 * b_pred_01   # {-1,+1}

        C_train_new = interpose(C_train.astype(np.float64),
                                b_pred.reshape(-1, 1), pos)
        Phi_new     = transform_atf(C_train_new)

        model_down = train_mlp_puf(
            X          = Phi_new,
            y          = y_train,
            net        = net,
            lr         = lr,
            epochs     = epochs,
            bs         = bs,
            seed       = 44 + rnd,
            early_stop = None,   # no early stop for lower retrain in rounds
        )

        # Evaluate lower model vs puf.down
        b_real_rnd  = puf.up.eval(C_test)
        C_star_rnd  = interpose(C_test.astype(np.float64),
                                b_real_rnd.reshape(-1, 1), pos)
        r_down_rnd  = puf.down.eval(C_star_rnd.astype(np.int8))
        y_down_rnd  = ((1 - r_down_rnd) // 2).astype(np.int8)
        Phi_lo_rnd  = transform_atf(C_star_rnd)
        X_lo_t      = torch.tensor(Phi_lo_rnd, dtype=torch.float32)
        y_lo_t      = torch.tensor(y_down_rnd, dtype=torch.float32)
        acc_lo_rnd  = evaluate_mlp(model_down, X_lo_t, y_lo_t, device)
        acc_lo_rnd  = max(acc_lo_rnd, 1 - acc_lo_rnd)
        print(f"  Lower model accuracy (vs puf.down): {acc_lo_rnd*100:.2f}%")

        # Full iPUF accuracy
        acc_total = ipuf_accuracy_mlp(puf, model_down, model_up, pos, n_bits, device)
        print(f"  Full iPUF accuracy   : {acc_total*100:.2f}%")

        if acc_total >= target_acc:
            print(f"\n  ✓ Target {target_acc*100:.0f}% reached at round {rnd+1}!")
            break

    print(f"\n{'─'*60}")
    if model_up is not None:
        final_acc = ipuf_accuracy_mlp(puf, model_down, model_up, pos, n_bits,
                                       device, n_eval=20_000)
        print(f"  Final iPUF accuracy  : {final_acc*100:.2f}%")
    else:
        print("  Attack did not reach upper model training stage.")
    print(f"{'─'*60}\n")


# ──────────────────────────────────────────────────────────────────────────────
#  7.  Main
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    num_bits = 128
    k_up     = 5
    k_lo     = 5
    puf_id   = 0
    N_TRAIN  = 2_000_000
    N_TEST   =  40_000

    puf = InterposePUF(
        n=num_bits,
        k_up=k_up,
        k_down=k_lo,
        seed=1 + puf_id,
        noisiness=0,
    )

    split_attack_mlp(
        puf        = puf,
        n_bits     = num_bits,
        k_up       = k_up,
        k_lo       = k_lo,
        n_train    = N_TRAIN,
        n_test     = N_TEST,
        max_rounds = 5,
        target_acc = 0.95,
        puf_id     = puf_id,
        net        = [256, 128, 64],
        lr         = 0.001,
        epochs     = 30,
        bs         = 1000,
        early_stop = 0.74,
    )

In [ ]:
"""
PC-LPUF Split Attack — MLP version
====================================
Correct attack logic from pclpuf_split_attack_final.py,
with LR replaced by PyTorch MLP throughout.

Requirements:
    pip install numpy torch scikit-learn scipy
"""

import numpy as np
from itertools import product as iterproduct
from sklearn.metrics import accuracy_score
from scipy import sparse
import csv, os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# ──────────────────────────────────────────────────────────────────────────────
#  1.  ReapNVM PUF
# ──────────────────────────────────────────────────────────────────────────────
def ReapNVM(num_bits, seed, sigma_proc = 0.05):
    rng = np.random.default_rng(seed)

    chal_length = num_bits
    n_levels = 4

    # ---- Fixed nominal resistance levels ----
    R_levels_nom = np.array([10e3, 75e3, 125e3, 275e3])

    # ---- Convert to log domain ----
    log_R_levels_nom = np.log10(R_levels_nom)

    # ---- Allocate output arrays ----
    tR = np.zeros((2, n_levels, chal_length))

    # ---- Generate per-cell values ----
    for row in range(2):
        for stage in range(chal_length):

            # Add Gaussian process variation in log domain
            log_levels = log_R_levels_nom + sigma_proc * rng.standard_normal(n_levels)

            # Convert back to linear domain
            levels = 10 ** log_levels

            # RC delay mapping
            tR[row, :, stage] = levels * 250e-12

    # ---- Deterministic switching delay ----
    tSW = np.full((2, 2, chal_length), 372.0 / 1e12)

    return 4.0 * tR, 4.0 * tSW

def ReapNVM_evaluate(PUF, challenge, position, value, chunk_size=100_000):
    chal      = (challenge + 1) / 2.0
    tR4, tSW4 = PUF
    chalpos   = position.astype(int).flatten()
    chalval   = value.astype(int).flatten()
    N         = chal.shape[0]
    responses = np.zeros(N, dtype=np.int8)
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        n   = end - start
        cc  = chal[start:end]; pc = chalpos[start:end]; vc = chalval[start:end]
        tv1 = np.tile(tR4[0, 0, :], (n, 1)); tv2 = np.tile(tR4[1, 0, :], (n, 1))
        tv1[np.arange(n), pc] = tR4[0, vc, pc]
        tv2[np.arange(n), pc] = tR4[1, vc, pc]
        c  = np.bitwise_xor.accumulate(cc.astype(np.uint8), axis=1)
        t1 = np.sum(np.where(c == 0, tv1 + tSW4[0, 0, :], tv2 + tSW4[1, 0, :]), axis=1)
        t2 = np.sum(np.where(c == 0, tv2 + tSW4[0, 1, :], tv1 + tSW4[1, 1, :]), axis=1)
        responses[start:end] = (t1 > t2).astype(np.int8)
    return responses


# ──────────────────────────────────────────────────────────────────────────────
#  2.  APUF
# ──────────────────────────────────────────────────────────────────────────────

def apuf_generate(k, chal_size, seed=0):
    return np.random.default_rng(seed).normal(0, 1, (k, chal_size + 1))

def apuf_response(w, Phi):
    return (Phi @ w <= 0).astype(np.int8)


# ──────────────────────────────────────────────────────────────────────────────
#  3.  XOR Obfuscation
# ──────────────────────────────────────────────────────────────────────────────

def dec_to_bin_vec(x, bitlen):
    return np.array([(x >> i) & 1 for i in range(bitlen)][::-1], dtype=np.uint8)

def bin_vec_to_dec(bits):
    out = 0
    for b in bits: out = (out << 1) | int(b)
    return out

def sliding_window_xor(bits, window_bits):
    window_bits = window_bits.astype(bits.dtype)
    for start in range(0, bits.shape[0], window_bits.shape[0]):
        end = min(start + window_bits.shape[0], bits.shape[0])
        bits[start:end] ^= window_bits[:end - start]
    return bits

def xor_obfuscate_position_value(position, value, upper_resp):
    N = position.shape[0]
    pos_out = np.zeros(N, dtype=np.uint32)
    val_out = np.zeros(N, dtype=np.uint32)
    for n in range(N):
        window_bits = upper_resp[:, n]
        pos_bits    = sliding_window_xor(dec_to_bin_vec(position[n], 7), window_bits)
        val_bits    = sliding_window_xor(dec_to_bin_vec(value[n],    2), window_bits)
        pos_out[n]  = bin_vec_to_dec(pos_bits)
        val_out[n]  = bin_vec_to_dec(val_bits)
    return pos_out, val_out


# ──────────────────────────────────────────────────────────────────────────────
#  4.  PC-LPUF Evaluate
# ──────────────────────────────────────────────────────────────────────────────

def pclpuf_evaluate(upper_w, lower_pufs, challenges, position, value):
    K_UP = upper_w.shape[0]
    N    = challenges.shape[0]
    Phi  = transform(challenges)
    upper_resp = np.array([apuf_response(upper_w[i], Phi) for i in range(K_UP)])
    pos_eff, val_eff = xor_obfuscate_position_value(position, value, upper_resp)
    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        xor_resp = np.bitwise_xor(xor_resp, ReapNVM_evaluate(puf, challenges, pos_eff, val_eff))
    return xor_resp

def lower_layer_evaluate(lower_pufs, challenges, pos_eff, val_eff):
    N        = challenges.shape[0]
    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        xor_resp = np.bitwise_xor(xor_resp, ReapNVM_evaluate(puf, challenges, pos_eff, val_eff))
    return xor_resp


# ──────────────────────────────────────────────────────────────────────────────
#  5.  Feature Transform
# ──────────────────────────────────────────────────────────────────────────────

def transform(challenges):
    N = challenges.shape[0]
    return np.hstack([np.cumprod(challenges, axis=1), np.ones((N, 1))])

def prepare_lr_features(Phi, positions, values, chal_size, n_levels):
    """Build feature matrix — returns dense numpy array for MLP."""
    N         = Phi.shape[0]
    pos       = positions.astype(int)
    val       = values.astype(int)
    delta_idx = pos * n_levels + val
    data      = Phi[np.arange(N), pos]
    X_delta   = sparse.csr_matrix((data, (np.arange(N), delta_idx)),
                                   shape=(N, chal_size * n_levels))
    return sparse.hstack([sparse.csr_matrix(Phi), X_delta], format='csr').toarray()


# ──────────────────────────────────────────────────────────────────────────────
#  6.  MLP  (user's exact code)
# ──────────────────────────────────────────────────────────────────────────────

def train_mlp_puf(X, y, net=[128, 32, 16], lr=0.001, epochs=20, bs=1000,
                  seed=1, early_stop=None):
    torch.manual_seed(seed)
    np.random.seed(seed)
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_t     = torch.tensor(X, dtype=torch.float32)
    y_t     = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    dataset = TensorDataset(X_t, y_t)
    loader  = DataLoader(dataset, batch_size=bs, shuffle=True)
    layers  = []; in_dim = X.shape[1]
    for h in net:
        layers += [nn.Linear(in_dim, h), nn.ReLU()]; in_dim = h
    layers += [nn.Linear(in_dim, 1), nn.Sigmoid()]
    model   = nn.Sequential(*layers).to(device)
    opt     = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    for epoch in range(epochs):
        model.train(); total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()
        acc = evaluate_mlp(model, X_t, y_t, device)
        print(f"  Epoch {epoch+1}/{epochs} | Loss={total_loss:.4f} | Acc={acc:.4f}")
        if early_stop is not None and acc >= early_stop:
            print("  Early stop triggered."); break
    return model

def evaluate_mlp(model, X_t, y_t, device, batch_size=2048):
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb    = torch.tensor(X_t[i:i+batch_size], dtype=torch.float32).to(device)
            yb    = torch.tensor(y_t[i:i+batch_size], dtype=torch.float32).view(-1, 1)
            preds = (model(xb).cpu() > 0.5).int()
            preds = 2 * preds - 1
            yb    = 2 * yb.int() - 1
            correct += (preds == yb).sum().item(); total += yb.size(0)
    return correct / total

def predict_mlp(model, X, device, batch_size=2048):
    """Return {0,1} predictions."""
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            preds.append((model(xb).cpu().numpy() > 0.5).astype(np.int8))
    return np.vstack(preds).flatten()


# ──────────────────────────────────────────────────────────────────────────────
#  7.  Phase 1 — Random upper bit guess
# ──────────────────────────────────────────────────────────────────────────────

def interpose_random_pclpuf(challenges, position, value,
                             K_UP, chal_size, n_levels, seed=0):
    N   = challenges.shape[0]
    rng = np.random.default_rng(seed)
    Phi = transform(challenges)
    guessed_upper = rng.integers(0, 2, size=(K_UP, N)).astype(np.int8)
    pos_chosen, val_chosen = xor_obfuscate_position_value(position, value, guessed_upper)
    return prepare_lr_features(Phi, pos_chosen, val_chosen, chal_size, n_levels)


# ──────────────────────────────────────────────────────────────────────────────
#  8.  Build Upper PUF Training Set
# ──────────────────────────────────────────────────────────────────────────────

def build_upper_training_set(challenges, position, value, y_pclpuf,
                              lower_model, chal_size, n_levels, K_UP,
                              device, block_size=10_000, max_combos=None):
    all_combos = np.array(list(iterproduct([0, 1], repeat=K_UP)), dtype=np.int8)
    if max_combos is not None and max_combos < len(all_combos):
        idx    = np.random.choice(len(all_combos), max_combos, replace=False)
        combos = all_combos[idx]
        print(f"    Using {max_combos}/{len(all_combos)} combos")
    else:
        combos = all_combos
        print(f"    Using all {len(all_combos)} combos")

    N         = challenges.shape[0]
    sel_chals = []; sel_resps = []

    for idx in range(0, N, block_size):
        bc  = challenges[idx:idx+block_size]
        bp  = position[idx:idx+block_size]
        bv  = value[idx:idx+block_size]
        br  = y_pclpuf[idx:idx+block_size]
        bn  = len(bc); phi = transform(bc)

        all_preds = np.zeros((len(combos), bn), dtype=np.int8)
        for c_idx, combo in enumerate(combos):
            ur           = np.tile(combo.reshape(K_UP, 1), (1, bn))
            pos_c, val_c = xor_obfuscate_position_value(bp, bv, ur)
            X_c          = prepare_lr_features(phi, pos_c, val_c, chal_size, n_levels)
            all_preds[c_idx] = predict_mlp(lower_model, X_c, device)

        matches = (all_preds == br[np.newaxis, :])

        n_insensitive = 0; n_model_wrong = 0
        for n in range(bn):
            correct_idxs = np.where(matches[:, n])[0]
            if len(correct_idxs) == 0:
                n_model_wrong += 1; continue
            if len(correct_idxs) == len(combos):
                n_insensitive += 1
            chosen_combo = combos[correct_idxs[np.random.randint(len(correct_idxs))]]
            sel_chals.append(bc[n]); sel_resps.append(chosen_combo)

        if (idx + block_size) % 50_000 == 0:
            print(f"    {min(idx+block_size, N)}/{N} processed, "
                  f"{len(sel_chals)} selected | "
                  f"insensitive={n_insensitive} model_wrong={n_model_wrong}")

    if len(sel_chals) == 0: return None, None
    return np.array(sel_chals), np.array(sel_resps)


# ──────────────────────────────────────────────────────────────────────────────
#  9.  Full Split Attack
# ──────────────────────────────────────────────────────────────────────────────

def split_attack(upper_w, lower_pufs, chal_size, n_levels, K_UP, K_d,
                 n_train, n_test, max_rounds=5, target_acc=0.90,
                 net=[256, 128, 64], lr=0.001, epochs=30, bs=1000,
                 max_combos=None):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{'='*60}")
    print(f"  PC-LPUF Split Attack — MLP")
    print(f"  chal_size={chal_size}, K_UP={K_UP}, K_d={K_d}")
    print(f"  CRPs : train={n_train:,}  test={n_test:,}")
    print(f"{'='*60}")

    # ── Generate CRPs ─────────────────────────────────────────────────────────
    print("\n[1] Generating CRPs...")
    np.random.seed(60)
    challenges_tr = np.random.choice([-1, 1], size=(n_train, chal_size))
    pos_tr        = np.random.randint(0, chal_size, n_train)
    val_tr        = np.random.randint(0, n_levels,  n_train)
    challenges_te = np.random.choice([-1, 1], size=(n_test,  chal_size))
    pos_te        = np.random.randint(0, chal_size, n_test)
    val_te        = np.random.randint(0, n_levels,  n_test)

    y_train = pclpuf_evaluate(upper_w, lower_pufs, challenges_tr, pos_tr, val_tr)
    y_test  = pclpuf_evaluate(upper_w, lower_pufs, challenges_te, pos_te, val_te)
    print(f"    Response balance: {y_train.mean()*100:.1f}% ones")

    Phi_te       = transform(challenges_te)
    y_lower_true = lower_layer_evaluate(lower_pufs, challenges_te, pos_te, val_te)

    # ── Phase 1: Train lower MLP with random upper bits ───────────────────────
    print(f"\n[2] Phase 1 — Training lower MLP (early_stop=0.74)...")

    lower_model = None; acc_lo = 0.0

    for attempt in range(10):
        print(f"\n    Attempt {attempt+1}/10")
        base_seed = 271828 + attempt

        X_train = interpose_random_pclpuf(
            challenges_tr, pos_tr, val_tr, K_UP, chal_size, n_levels,
            seed=30 + base_seed)

        lower_model = train_mlp_puf(
            X=X_train, y=y_train, net=net, lr=lr,
            epochs=epochs, bs=bs, seed=base_seed, early_stop=0.74)

        # Retry check: random bit vs y_pclpuf
        X_check   = interpose_random_pclpuf(
            challenges_te, pos_te, val_te, K_UP, chal_size, n_levels,
            seed=50 + base_seed)
        X_t_check = torch.from_numpy(X_check.astype(np.float32))
        y_t_check = torch.from_numpy(y_test.astype(np.float32))
        acc_check = evaluate_mlp(lower_model, X_t_check, y_t_check, device)
        acc_check = max(acc_check, 1 - acc_check)
        print(f"    Attempt {attempt+1} accuracy (random bit vs y_pclpuf): {acc_check*100:.2f}%")

        # Debug: random bit vs real lower layer
        rng_dbg  = np.random.default_rng(70 + base_seed)
        g_upper  = rng_dbg.integers(0, 2, size=(K_UP, len(challenges_te))).astype(np.int8)
        pos_d, val_d = xor_obfuscate_position_value(pos_te, val_te, g_upper)
        y_lo_rnd = lower_layer_evaluate(lower_pufs, challenges_te, pos_d, val_d)
        X_debug  = prepare_lr_features(Phi_te, pos_d, val_d, chal_size, n_levels)
        X_t_dbg  = torch.from_numpy(X_debug.astype(np.float32))
        y_t_dbg  = torch.from_numpy(y_lo_rnd.astype(np.float32))
        acc_lo   = evaluate_mlp(lower_model, X_t_dbg, y_t_dbg, device)
        acc_lo   = max(acc_lo, 1 - acc_lo)
        print(f"    Attempt {attempt+1} lower layer acc (random bit vs puf.down) [debug]: {acc_lo*100:.2f}%")

        if not (0.45 <= acc_check <= 0.55):
            print(f"    ✓ Phase 1 succeeded at attempt {attempt+1}"); break
        else:
            print(f"    ✗ Stuck at ~50%, retrying...")

    # ── Iterative refinement ──────────────────────────────────────────────────
    upper_models = None; acc_lo2 = acc_lo; acc_full = 0.0; acc_up_list = []

    for rnd in range(max_rounds):
        print(f"\n[Round {rnd+1}]")

        # Build upper training set
        print(f"  Building upper training set ({2**K_UP} combos × {n_train:,} challenges)...")
        sel_C, sel_r = build_upper_training_set(
            challenges_tr, pos_tr, val_tr, y_train,
            lower_model, chal_size, n_levels, K_UP, device,
            max_combos=max_combos)

        if sel_C is None or len(sel_C) < 50:
            print("  WARNING: Not enough challenges. Stopping."); break

        print(f"  Selected {len(sel_C):,} ({len(sel_C)/n_train*100:.1f}%)")

        # Train K_UP independent APUF MLPs — print after each
        Phi_up_sel    = transform(sel_C)
        upper_resp_te = np.array([apuf_response(upper_w[i], Phi_te) for i in range(K_UP)])
        upper_models  = []; acc_up_list = []

        for i in range(K_UP):
            print(f"  Training upper APUF {i+1}/{K_UP}...")
            model_i = train_mlp_puf(
                X=Phi_up_sel, y=sel_r[:, i],
                net=net, lr=lr, epochs=epochs, bs=bs,
                seed=43 + rnd + i, early_stop=None)
            X_t_up = torch.from_numpy(Phi_te.astype(np.float32))
            y_t_up = torch.from_numpy(upper_resp_te[i].astype(np.float32))
            acc_i  = evaluate_mlp(model_i, X_t_up, y_t_up, device)
            acc_i  = max(acc_i, 1 - acc_i)
            acc_up_list.append(acc_i); upper_models.append(model_i)
            print(f"  ✓ Upper APUF {i+1}/{K_UP} done — accuracy: {acc_i*100:.2f}%")

        print(f"  Upper mean accuracy: {np.mean(acc_up_list)*100:.2f}%")

        # Retrain lower with predicted upper bits
        print("  Retraining lower model with predicted upper bits...")
        Phi_tr        = transform(challenges_tr)
        pred_upper_tr = np.array([predict_mlp(upper_models[i], Phi_tr, device)
                                   for i in range(K_UP)])
        pos_tr_pred, val_tr_pred = xor_obfuscate_position_value(pos_tr, val_tr, pred_upper_tr)
        X_train_new = prepare_lr_features(Phi_tr, pos_tr_pred, val_tr_pred, chal_size, n_levels)
        lower_model = train_mlp_puf(
            X=X_train_new, y=y_train, net=net, lr=lr,
            epochs=epochs, bs=bs, seed=44 + rnd, early_stop=None)

        # Evaluate lower model vs real lower layer
        X_te_lo = prepare_lr_features(Phi_te, pos_te, val_te, chal_size, n_levels)
        X_t_lo  = torch.from_numpy(X_te_lo.astype(np.float32))
        y_t_lo  = torch.from_numpy(y_lower_true.astype(np.float32))
        acc_lo2 = evaluate_mlp(lower_model, X_t_lo, y_t_lo, device)
        acc_lo2 = max(acc_lo2, 1 - acc_lo2)
        print(f"  Lower model accuracy (vs real lower layer): {acc_lo2*100:.2f}%")

        # Full PC-LPUF accuracy: predict upper → obfuscate → lower → vs y_pclpuf
        pred_upper_te = np.array([predict_mlp(upper_models[i], Phi_te, device)
                                   for i in range(K_UP)])
        pos_te_pred, val_te_pred = xor_obfuscate_position_value(pos_te, val_te, pred_upper_te)
        X_te_full = prepare_lr_features(Phi_te, pos_te_pred, val_te_pred, chal_size, n_levels)
        pred_full = predict_mlp(lower_model, X_te_full, device)
        acc_full  = max(accuracy_score(y_test, pred_full),
                        1 - accuracy_score(y_test, pred_full))
        print(f"  Full PC-LPUF accuracy: {acc_full*100:.2f}%")

        if acc_full >= target_acc:
            print(f"\n  ✓ Target {target_acc*100:.0f}% reached at round {rnd+1}!"); break

    # Final summary
    print(f"\n{'─'*60}")
    print(f"  Done.")
    print(f"  Final full PC-LPUF accuracy : {acc_full*100:.2f}%")
    print(f"  Final lower layer accuracy  : {acc_lo2*100:.2f}%")
    print(f"  Phase 1 lower layer accuracy: {acc_lo*100:.2f}%")
    print(f"{'─'*60}\n")

    # Save results
    results_file = f'pclpuf_results_K_UP{K_UP}_Kd{K_d}.csv'
    file_exists  = os.path.isfile(results_file)
    with open(results_file, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['K_UP','K_d','chal_size','n_levels','n_train','n_test',
                             'max_rounds','phase1_lower_acc','final_lower_acc',
                             'final_upper_mean_acc','final_full_acc'])
        writer.writerow([K_UP, K_d, chal_size, n_levels, n_train, n_test, max_rounds,
                         f'{acc_lo*100:.2f}', f'{acc_lo2*100:.2f}',
                         f'{np.mean(acc_up_list)*100:.2f}' if len(acc_up_list) > 0 else 'N/A',
                         f'{acc_full*100:.2f}'])
    print(f"  Results saved to {results_file}")


# ──────────────────────────────────────────────────────────────────────────────
#  10.  Main
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    chal_size = 128
    n_levels  = 4
    K_UP      = 5
    K_d       = 5
    n_train   = 2_000_000
    n_test    =  20_000

    np.random.seed(60)
    upper_w    = apuf_generate(K_UP, chal_size, seed=0)
    lower_pufs = [ReapNVM(chal_size, seed=42 + k) for k in range(K_d)]

    split_attack(
        upper_w    = upper_w,
        lower_pufs = lower_pufs,
        chal_size  = chal_size,
        n_levels   = n_levels,
        K_UP       = K_UP,
        K_d        = K_d,
        n_train    = n_train,
        n_test     = n_test,
        max_rounds = 5,
        target_acc = 0.90,
        net        = [256, 128, 64],
        lr         = 0.001,
        epochs     = 30,
        bs         = 1000,
        max_combos = None,  # None = all 2^K_UP, or set e.g. 8 for speed
    )


  PC-LPUF Split Attack — MLP
  chal_size=128, K_UP=5, K_d=5
  CRPs : train=2,000,000  test=20,000

[1] Generating CRPs...
    Response balance: 50.0% ones

[2] Phase 1 — Training lower MLP (early_stop=0.74)...

    Attempt 1/10


/tmp/ipykernel_2627510/3741147912.py:203: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xb    = torch.tensor(X_t[i:i+batch_size], dtype=torch.float32).to(device)
/tmp/ipykernel_2627510/3741147912.py:204: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  yb    = torch.tensor(y_t[i:i+batch_size], dtype=torch.float32).view(-1, 1)


  Epoch 1/30 | Loss=1386.2754 | Acc=0.5064
  Epoch 2/30 | Loss=1385.9400 | Acc=0.5137
  Epoch 3/30 | Loss=1384.9581 | Acc=0.5252
  Epoch 4/30 | Loss=1383.2626 | Acc=0.5337
  Epoch 5/30 | Loss=1380.4313 | Acc=0.5420
  Epoch 6/30 | Loss=1376.4181 | Acc=0.5545
  Epoch 7/30 | Loss=1371.3033 | Acc=0.5630
  Epoch 8/30 | Loss=1365.4944 | Acc=0.5709
  Epoch 9/30 | Loss=1359.2622 | Acc=0.5796
  Epoch 10/30 | Loss=1352.7777 | Acc=0.5868
  Epoch 11/30 | Loss=1346.3386 | Acc=0.5921
  Epoch 12/30 | Loss=1339.9115 | Acc=0.5985
  Epoch 13/30 | Loss=1333.4999 | Acc=0.6037
  Epoch 14/30 | Loss=1327.1446 | Acc=0.6079
  Epoch 15/30 | Loss=1321.1449 | Acc=0.6124
  Epoch 16/30 | Loss=1315.3692 | Acc=0.6162
  Epoch 17/30 | Loss=1309.7244 | Acc=0.6202
  Epoch 18/30 | Loss=1304.8483 | Acc=0.6234
  Epoch 19/30 | Loss=1299.8277 | Acc=0.6260
  Epoch 20/30 | Loss=1294.8920 | Acc=0.6287
  Epoch 21/30 | Loss=1290.3880 | Acc=0.6304
  Epoch 22/30 | Loss=1286.1654 | Acc=0.6340
  Epoch 23/30 | Loss=1281.8606 | Acc=0.63